# Group 20 — A: data preparation for ETA prediction

This is A's data module, not the complete group submission. Local development uses the complete January 2026 Yellow Taxi file. Re-run the agreed scope on Rockhill before using results as assessment evidence.

Target: duration in minutes. Prediction setting: at pickup, assuming intended destination zone is known. Actual recorded dropoff zone is a proxy for this assumption. The cleaned population excludes fixed invalid/extreme-trip cases; report this selection limitation.


In [1]:
from pathlib import Path
import sys, json
if 'DELIVERY_ROOT' in globals():
    ROOT = Path(DELIVERY_ROOT)
else:
    candidates = [Path.cwd(), Path.cwd() / 'CS4207/Group20_ETA_A', Path.cwd() / 'Group20_ETA_A']
    ROOT = next((p for p in candidates if (p / 'eta_data.py').exists()), None)
    if ROOT is None:
        raise FileNotFoundError('Open this notebook in the Group20_ETA_A folder.')
sys.path.insert(0, str(ROOT))
from eta_data import read_config, run_pipeline, RULES
from handoff import load_sample, chronological_split, inputs_and_target
import pandas as pd
config, settings = read_config(ROOT / 'config.json')
files = sorted((ROOT / 'data/raw').glob('yellow_tripdata_2026-01.parquet'))
if not files:
    raise FileNotFoundError('Place the raw January file in data/raw, or point files to the agreed Rockhill data files.')
print('Input files:', [p.name for p in files])
print('Config:', config)


Input files: ['yellow_tripdata_2026-01.parquet']
Config: CleaningConfig(batch_size=100000, sample_size=100000, sample_seed=420720, min_duration_minutes=1.0, max_duration_minutes=180.0, max_distance_miles=100.0, max_speed_mph=100.0)


## 1. Re-run the actual cleaning code

The following cell executes the full cleaning pipeline again, rather than displaying invented or cached counts. Existing exports are only reused after their fingerprint is checked. For a fresh export, set `export_path` to a new path and run `run_pipeline(files, config, workers=1, output=export_path)`.


In [2]:
export_path = ROOT / 'outputs/local_jan2026'
if not (export_path / 'manifest.json').exists():
    result = run_pipeline(files, config, workers=1, output=export_path)
else:
    result = run_pipeline(files, config, workers=1)
    saved = json.loads((export_path / 'manifest.json').read_text(encoding='utf-8'))
    assert result['ordered_result_sha256'] == saved['result']['ordered_result_sha256'], 'Raw data/config/code changed: make a fresh export.'
print(json.dumps(result, indent=2))


{
  "workers": 1,
  "timed_cleaning_seconds": 4.266821200000777,
  "wall_seconds_including_exports": 4.266844599998876,
  "rows_in": 3724889,
  "rows_kept": 3504372,
  "flex_fare_rows_in": 1088058,
  "flex_fare_rows_kept": 993400,
  "missing_passenger_rows_in": 1088058,
  "missing_passenger_rows_kept": 993400,
  "missing_rate_rows_in": 1088058,
  "missing_rate_rows_kept": 993400,
  "invalid_passenger_values_to_missing": 0,
  "removed": {
    "missing_timestamps": 0,
    "pickup_outside_file_month": 7,
    "non_positive_duration": 45070,
    "duration_outside_scope": 40430,
    "missing_or_nonfinite_distance": 0,
    "non_positive_distance": 98392,
    "distance_above_limit": 156,
    "missing_or_nonfinite_fare": 0,
    "non_positive_fare": 36213,
    "speed_above_limit": 249
  },
  "ordered_result_sha256": "79d5ca63db72c4a5f1f311652498d04fc22397439fab807fa899f5364d6d5cfe"
}


## 2. Ordered cleaning audit

Rows are removed at their first failing rule, so removal counts do not overlap. Thresholds (1–180 minutes, at most 100 miles, at most 100 mph) are explicit project choices, not lecturer-prescribed constants. Do not drop rows simply because passenger count or rate code is missing.


In [3]:
remaining = result['rows_in']
rows = [{'rule': 'Input rows', 'removed': 0, 'remaining': remaining}]
for key, label in RULES:
    remaining -= result['removed'][key]
    rows.append({'rule': label, 'removed': result['removed'][key], 'remaining': remaining})
audit = pd.DataFrame(rows)
print(audit.to_string(index=False))
assert remaining == result['rows_kept']
print('Kept Flex Fare rows:', result['flex_fare_rows_kept'])


                                             rule  removed  remaining
                                       Input rows        0    3724889
      Missing or invalid pickup/dropoff timestamp        0    3724889
      Pickup outside the month in source filename        7    3724882
                            Duration <= 0 minutes    45070    3679812
   Duration outside configured [min, max] minutes    40430    3639382
                   Distance missing or non-finite        0    3639382
                              Distance <= 0 miles    98392    3540990
              Distance exceeds configured maximum      156    3540834
                       Fare missing or non-finite        0    3540834
                                Fare <= 0 dollars    36213    3504621
Observed average speed exceeds configured maximum      249    3504372
Kept Flex Fare rows: 993400


## 3. Deterministic development sample and schema

100,000 retained trips are selected across the whole month using a fixed row-key hash and seed, then sorted by pickup time. This is not the final test set. Stable row IDs preserve alignment across parallel feature tasks. Full exported parts are not globally chronological: downstream splitting must use timestamps.


In [4]:
sample = load_sample(export_path / 'sample_100k.parquet')
print('Sample rows:', len(sample))
print(sample.head(5).to_string(index=False))
print(sample.dtypes.to_string())
print('Missing values:\n', sample.isna().sum().to_string())
assert sample['row_id'].is_unique
assert sample['pickup_datetime'].is_monotonic_increasing


Sample rows: 100000
                                      row_id                     source_file     pickup_datetime  duration_minutes pickup_zone dropoff_zone  passenger_count  passenger_count_missing   rate_code vendor_id
yellow_tripdata_2026-01.parquet:000000003050 yellow_tripdata_2026-01.parquet 2026-01-01 00:02:26         18.400000         164          143              1.0                    False           1         2
yellow_tripdata_2026-01.parquet:000000001219 yellow_tripdata_2026-01.parquet 2026-01-01 00:03:11          3.250000         249          249              1.0                    False           1         2
yellow_tripdata_2026-01.parquet:000002639809 yellow_tripdata_2026-01.parquet 2026-01-01 00:03:42         16.983333         256          148              NaN                     True __MISSING__         2
yellow_tripdata_2026-01.parquet:000000000825 yellow_tripdata_2026-01.parquet 2026-01-01 00:05:00          4.616667         237          229              1.0        

## 4. B/C handoff and leakage exclusions

These January boundaries are an integration example, not a team-approved final split. Change them together when agreeing data scope. Fit imputation, category encoders, scaling and model parameters on training data only. `rate_code` is a final-at-end-of-trip audit category; never auto-include it. Raw distance, speed, fare, tip, payment and dropoff time are not exported as model inputs.


In [5]:
train, validation, test = chronological_split(sample, '2026-01-22', '2026-01-27')
X_train_raw, y_train = inputs_and_target(train, intended_destination_known=True)
print('Illustrative split sizes:', len(train), len(validation), len(test))
print('Allowed raw X columns:', list(X_train_raw.columns))
print('Target:', y_train.name)
assert train['pickup_datetime'].max() < validation['pickup_datetime'].min()
assert validation['pickup_datetime'].max() < test['pickup_datetime'].min()


Illustrative split sizes: 67074 14415 18511
Allowed raw X columns: ['pickup_datetime', 'pickup_zone', 'dropoff_zone', 'passenger_count', 'passenger_count_missing']
Target: duration_minutes


## 5. Local data-parallel benchmark only

One full warm-up per worker configuration, then at least three repeats of identical work. This measures raw reading, cleaning, ordered aggregation and output hashing; it is not training speedup. Local Windows results must not be labelled Rockhill results. This module does not complete the group's two task-parallel patterns or two observed nondeterminism/fix experiments.


In [6]:
benchmark_path = ROOT / 'outputs/local_benchmark/benchmark_summary.csv'
if benchmark_path.exists():
    print(pd.read_csv(benchmark_path).to_string(index=False))
else:
    print('Run: python eta_data.py --benchmark --output outputs/local_benchmark --repeats 3')
print('Server command: srun --partition=cs4207 python eta_data.py --input-glob "/home/cs4207/data/yellow_tripdata_2026-01.parquet" --output outputs/rockhill_run --workers 1')


 workers  mean_seconds  std_seconds  speedup  efficiency  all_outputs_equal
       1      4.553259     0.058418 1.000000    1.000000               True
       2      4.272246     0.181916 1.065777    0.532888               True
       4      3.513110     0.112689 1.296077    0.324019               True
Server command: srun --partition=cs4207 python eta_data.py --input-glob "/home/cs4207/data/yellow_tripdata_2026-01.parquet" --output outputs/rockhill_run --workers 1
